<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/23_query_normalization/query_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers scikit-learn --quiet

In [ ]:
import re

In [ ]:
def normalize_query(query):
    query = query.lower()
    query = re.sub(r"[^\w\s]", "", query)  # remove punctuation
    query = query.strip()
    return query

In [10]:
queries = [
    "What is capital of France?",
    "capital france?",
    "Tell me France capital!!!",
    "france capital??"
]

for q in queries:
    print("Original:", q)
    print("Normalized:", normalize_query(q))
    print("-" * 40)

Original: What is capital of France?
Normalized: what is capital of france
----------------------------------------
Original: capital france?
Normalized: capital france
----------------------------------------
Original: Tell me France capital!!!
Normalized: tell me france capital
----------------------------------------
Original: france capital??
Normalized: france capital
----------------------------------------


In [ ]:
# -------------------------------
# RAG PIPELINE (SELF-CONTAINED)
# -------------------------------

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Knowledge base
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

# TF-IDF setup
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Embeddings
embeddings = embed_model.encode(documents)


# -------------------------------
# RAG FUNCTION
# -------------------------------

def rag_pipeline(query):

    # Step 1: Retrieval (Hybrid)
    query_embedding = embed_model.encode([query])
    query_tfidf = vectorizer.transform([query])

    semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
    keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

    hybrid_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    best_index = hybrid_scores.argmax()
    context = documents[best_index]

    # Step 2: Answer Generation
    prompt = f"""
You are a question answering system.

Use ONLY the context below to answer.

Context: {context}

Question: {query}

Answer in one short sentence.
"""

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [ ]:
def rag_pipeline_normalized(query):
    clean_query = normalize_query(query)
    return rag_pipeline(clean_query)

In [11]:
query_variations = [
    "What is capital of France?",
    "capital france?",
    "france capital??"
]

for q in query_variations:
    print("Query:", q)
    print("Answer:", rag_pipeline_normalized(q))
    print("=" * 50)

Query: What is capital of France?
Answer: Paris
Query: capital france?
Answer: Paris
Query: france capital??
Answer: Paris
